## BONUS: Live Computer Vision with YOLO

Now let's see machine learning in action! We'll use a state-of-the-art YOLO (You Only Look Once) model to detect objects in real-time using your webcam.

**What is YOLO?**
- YOLO is a real-time object detection system
- It can identify and locate multiple objects in images/video
- Used in self-driving cars, security systems, and mobile apps
- Represents the cutting edge of computer vision

**Note:** This section demonstrates advanced ML concepts that we'll cover in Week 4. For now, just enjoy the demonstration!

In [ ]:
# Import required libraries for computer vision
import cv2
from ultralytics import YOLO
import time
import threading

# Check if webcam is available (with timeout to prevent hanging)
def check_webcam(timeout=5):
    """Check if webcam is available, with a timeout so it never hangs"""
    result = [False]
    cap_holder = [None]
    
    def try_open():
        cap = cv2.VideoCapture(0)
        if cap.isOpened():
            result[0] = True
            cap_holder[0] = cap
    
    thread = threading.Thread(target=try_open)
    thread.daemon = True
    thread.start()
    thread.join(timeout=timeout)
    
    if result[0]:
        print("✓ Webcam detected and ready!")
        if cap_holder[0]:
            cap_holder[0].release()
        return True
    else:
        print("✗ No webcam detected (or timed out). Will use sample image instead.")
        return False

# Test webcam availability
webcam_available = check_webcam()

print("\nSetting up YOLO object detection...")
print("This may take a moment to download the model weights...")

# Load pre-trained YOLO model
try:
    model_yolo = YOLO('yolov8n.pt')
    print("✓ YOLO model loaded successfully!")
    print(f"Model: YOLOv8 Nano")
    print(f"Classes: {len(model_yolo.names)} object types can be detected")
    
    print("Example detectable objects:")
    example_classes = ['person', 'car', 'dog', 'cat', 'bottle', 'chair', 'laptop', 'phone']
    for cls in example_classes:
        if cls in model_yolo.names.values():
            print(f"  - {cls.title()}")
    
except Exception as e:
    print(f"Error loading YOLO model: {e}")
    print("Please ensure you have internet connection for model download.")
    model_yolo = None

In [ ]:
import matplotlib.pyplot as plt

# Live webcam object detection function
def run_live_detection(duration=30):
    """
    Run live object detection using webcam
    
    Args:
        duration: How long to run detection (seconds)
    """
    if not webcam_available or model_yolo is None:
        print("Webcam or YOLO model not available.")
        return
    
    print(f"Starting live object detection for {duration} seconds...")
    print("Press 'q' to quit early, or 'c' to capture a frame")
    print("Objects will be detected and labeled in real-time!")
    
    # Initialize webcam
    cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    cap.set(cv2.CAP_PROP_FPS, 30)
    
    start_time = time.time()
    frame_count = 0
    
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                print("Failed to capture frame")
                break
            
            # Run YOLO detection
            results = model_yolo(frame, verbose=False)
            annotated_frame = results[0].plot()
            
            # Add performance info
            frame_count += 1
            fps = frame_count / (time.time() - start_time)
            cv2.putText(annotated_frame, f'FPS: {fps:.1f}', (10, 30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            
            # Display frame
            cv2.imshow('YOLO Live Object Detection', annotated_frame)
            
            # Check for key presses
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                print("Quit requested by user")
                break
            elif key == ord('c'):
                filename = f'captured_frame_{int(time.time())}.jpg'
                cv2.imwrite(filename, annotated_frame)
                print(f"Frame captured: {filename}")
            
            # Check time limit
            if time.time() - start_time > duration:
                print(f"Time limit ({duration}s) reached")
                break
                
    except KeyboardInterrupt:
        print("Detection stopped by user")
    finally:
        cap.release()
        cv2.destroyAllWindows()
        
        total_time = time.time() - start_time
        avg_fps = frame_count / total_time if total_time > 0 else 0
        print(f"\nDetection Summary:")
        print(f"  Total time: {total_time:.1f} seconds")
        print(f"  Frames processed: {frame_count}")
        print(f"  Average FPS: {avg_fps:.1f}")


# Fallback: detect objects in a sample image
def run_image_detection():
    """Run object detection on a sample image"""
    if model_yolo is None:
        print("YOLO model not available.")
        return
        
    print("Running YOLO detection on sample image...")
    
    try:
        import urllib.request
        sample_image_url = "https://ultralytics.com/images/bus.jpg"
        urllib.request.urlretrieve(sample_image_url, "sample_image.jpg")
        
        results = model_yolo("sample_image.jpg")
        annotated_image = results[0].plot()
        annotated_image_rgb = cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB)
        
        plt.figure(figsize=(12, 8))
        plt.imshow(annotated_image_rgb)
        plt.axis('off')
        plt.title('YOLO Object Detection Results')
        plt.tight_layout()
        plt.show()
        
        print("\nDetection Results:")
        for result in results:
            boxes = result.boxes
            if boxes is not None:
                for box in boxes:
                    class_id = int(box.cls[0])
                    confidence = float(box.conf[0])
                    class_name = model_yolo.names[class_id]
                    print(f"  Detected: {class_name} (confidence: {confidence:.2f})")
        
    except Exception as e:
        print(f"Error in image detection: {e}")
        
print("YOLO functions defined successfully!")

In [ ]:
# YOLO Object Detection Demo
print("YOLO Object Detection Demo")
print("=" * 50)

if model_yolo is None:
    print("Cannot run YOLO demo — model failed to load.")
elif webcam_available:
    print("Webcam detected! Starting live detection for 30 seconds...")
    print("Press 'q' to quit early, 'c' to capture a frame\n")
    run_live_detection(duration=30)
else:
    print("No webcam available — running sample image detection...\n")
    run_image_detection()

print("\nYOLO Demo Complete!")
print("You just experienced state-of-the-art computer vision!")
print("In Week 4, we'll learn how these models work and build our own.")